# YES Bias on Polymarket: An Empirical Analysis

**Date:** 2026-03-25
**Sprint:** `sprints/yes-bias/`
**Data:** 88 resolved single-market events from Polymarket (29,463 trades, $5.5M volume)
**Academic reference:** Deleep et al. (2026) — "How Wise is the Crowd? Bias and Edge in Prediction Markets"

---

## Hypotheses

| ID | Hypothesis | Pre-registered threshold | Result |
|----|-----------|------------------------|--------|
| **H1** | Traders prefer buying YES tokens over NO tokens | YES buy proportion > 55%, p < 0.01 | **Not confirmed.** 51.3% — significant but below threshold |
| **H2** | YES tokens are systematically overpriced | Avg VWAP exceeds resolution rate by > 5pp | **Not confirmed.** Our sample shows underpricing (contradicts Deleep et al.) |
| **H3** | Small and large traders differ in YES/NO preference | YES buy % differs by > 10pp across size quartiles | **Confirmed on raw numbers.** 19pp gap — but primarily a price composition effect |

## Data

**88 resolved single-market events** from Polymarket's top 500 by volume. "Single-market" means one tradeable question per event — e.g., "Will TikTok be banned?" not "Presidential Election 2024" (which has 17 candidate markets inside one event). This distinction matters because multi-market events mechanically produce low YES resolution rates.

For each market we fetched up to 200 `OrderFilled` events per token (YES and NO) from the Goldsky subgraph. Trades are labeled BUY/SELL based on the **maker's** action (maker-centric classification).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

%matplotlib inline

sys.path.insert(0, str(Path("..").resolve()))
from src.analysis import (
    compute_trade_matrix, test_yes_vs_no_bias,
    compute_calibration, compute_market_vwap, compute_buyer_pnl,
)
from src.config import DATA_DIR, OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Load data
df = pd.read_parquet(str(DATA_DIR / "trades_single_market.parquet"))
df_multi = pd.read_parquet(str(DATA_DIR / "trades_all.parquet"))

print(f"Single-market dataset: {len(df):,} trades, {df['event'].nunique()} events, ${df['usdc'].sum():,.0f} volume")
print(f"Multi-market dataset:  {len(df_multi):,} trades, {df_multi['event'].nunique()} events")

---
## Section 1: Data Overview

Before testing hypotheses, we characterize the dataset so readers can assess whether our sample is adequate. This is not hypothesis testing — it's due diligence.

In [ ]:
from datetime import datetime

# Basic stats
ts_min = datetime.fromtimestamp(df["timestamp"].min())
ts_max = datetime.fromtimestamp(df["timestamp"].max())
market_stats = df.groupby("question").agg(
    trades=("timestamp", "count"),
    volume=("usdc", "sum"),
    resolved_yes=("resolved_yes", "first"),
    first_trade=("timestamp", "min"),
).reset_index()
market_stats["first_dt"] = market_stats["first_trade"].apply(datetime.fromtimestamp)

print(f"Markets: {len(market_stats)}")
print(f"Time range: {ts_min:%Y-%m-%d} to {ts_max:%Y-%m-%d}")
print(f"Resolved YES: {market_stats['resolved_yes'].sum()} / {len(market_stats)} ({market_stats['resolved_yes'].mean()*100:.1f}%)")
print(f"Trades per market: median={market_stats['trades'].median():.0f}, range=[{market_stats['trades'].min()}, {market_stats['trades'].max()}]")
print(f"Volume per market: median=${market_stats['volume'].median():,.0f}")

# Chart: distribution of trades per market and volume per market
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(market_stats["trades"], bins=30, color="#4ECDC4", edgecolor="white")
ax1.set_xlabel("Trades per market")
ax1.set_ylabel("Number of markets")
ax1.set_title("Trade count distribution")
ax1.axvline(market_stats["trades"].median(), color="#e74c3c", ls="--", label=f"Median: {market_stats['trades'].median():.0f}")
ax1.legend()

ax2.hist(market_stats["volume"], bins=30, color="#556270", edgecolor="white")
ax2.set_xlabel("Volume per market (USDC)")
ax2.set_ylabel("Number of markets")
ax2.set_title("Volume distribution")
ax2.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
ax2.axvline(market_stats["volume"].median(), color="#e74c3c", ls="--", label=f"Median: ${market_stats['volume'].median():,.0f}")
ax2.legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "01-data-overview.png"), dpi=150, bbox_inches="tight")

---
## Context: How Do Polymarket Events Actually Resolve?

This is not a hypothesis test — it's essential context. A commonly cited statistic claims ~80% of Polymarket markets resolve NO. If true, YES buying could be irrational. If not, we need to understand why the number is misleading before interpreting any bias results.

The key insight: the unit of analysis matters. An "event" on Polymarket is a container. A "market" is the tradeable unit. The Presidential Election 2024 is one event with 17 markets (one per candidate). When it resolves, 16 markets resolve NO and 1 resolves YES. That's not "94% NO resolution" — it's one event where one person won.

In [ ]:
# Single-market events: each event = exactly 1 market
single_res = df.groupby("event")["resolved_yes"].first()
yes_count = int(single_res.sum())
no_count = len(single_res) - yes_count

# Multi-market events: structural NO dominance
multi_only = df_multi[df_multi["num_markets_in_event"] > 1]
event_sizes = multi_only.groupby("event")["num_markets_in_event"].first()

print(f"=== Single-market events ===")
print(f"Resolved YES: {yes_count} / {len(single_res)} ({single_res.mean()*100:.1f}%)")
print(f"Resolved NO:  {no_count} / {len(single_res)} ({(~single_res).mean()*100:.1f}%)")
print(f"\n=== Multi-market events ===")
print(f"Events: {len(event_sizes)}, median markets per event: {event_sizes.median():.0f}")
print(f"Expected market-level YES rate: ~1/{event_sizes.median():.0f} = {1/event_sizes.median()*100:.1f}%")

# Chart: side-by-side resolution comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Single-market
ax1.bar(["YES", "NO"], [yes_count, no_count], color=["#4ECDC4", "#C44D58"])
ax1.set_title(f"Single-market events (n={len(single_res)})")
ax1.set_ylabel("Count")
for i, v in enumerate([yes_count, no_count]):
    ax1.text(i, v + 0.5, f"{v} ({v/len(single_res)*100:.0f}%)", ha="center", fontweight="bold")

# Multi-market structural explanation
ns = [5, 10, 20, 50]
rates = [1/n * 100 for n in ns]
ax2.bar([str(n) for n in ns], rates, color="#556270")
ax2.set_xlabel("Markets per event")
ax2.set_ylabel("YES resolution rate at market level (%)")
ax2.set_title("Multi-market events: structural NO dominance")
for i, (n, r) in enumerate(zip(ns, rates)):
    ax2.text(i, r + 0.5, f"{r:.0f}%", ha="center", fontweight="bold")
ax2.axhline(50, color="#e74c3c", ls="--", alpha=0.5, label="50%")
ax2.legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "02-resolution-rates.png"), dpi=150, bbox_inches="tight")

### Interpretation

Single-market events resolve YES 51% of the time — essentially a coin flip. The "80% NO" statistic comes from counting markets within multi-market events, where by construction only 1-of-N resolves YES.

**For our analysis this means:** the base rate for YES resolution is ~50%, not ~20%. Any YES buying bias we find is not explained by rational response to a high YES base rate, nor contradicted by a low one. The prior is neutral.

---
## H1: Do Traders Prefer Buying YES Over NO?

**Test:** Of all BUY trades across 88 markets, what proportion target YES vs NO tokens?
**Null hypothesis:** p(YES buy) = 0.5
**Pre-registered threshold:** YES buy proportion > 55% with p < 0.01
**Method:** Binomial test (aggregate) + market-level t-test

In [ ]:
# Aggregate binomial test
bias = test_yes_vs_no_bias(df)

# Market-level distribution
market_pcts = []
for q in df["question"].unique():
    m_buys = df[(df["question"] == q) & (df["side"] == "BUY")]
    if len(m_buys) < 20:
        continue
    market_pcts.append((m_buys["token_type"] == "YES").mean() * 100)

market_pcts = np.array(market_pcts)
t_stat, p_val = stats.ttest_1samp(market_pcts, 50)

print(f"=== Aggregate test ===")
print(f"YES buys: {bias['yes_buys']:,} / {bias['total_buys']:,} = {bias['yes_buy_pct']:.1f}%")
print(f"95% CI: [{bias['ci_95_low']:.1f}%, {bias['ci_95_high']:.1f}%]")
print(f"p-value: {bias['p_value']:.4f}")
print(f"\n=== Market-level test ({len(market_pcts)} markets) ===")
print(f"Mean YES buy %: {market_pcts.mean():.1f}%, std dev: {market_pcts.std():.1f}pp")
print(f"Range: [{market_pcts.min():.1f}%, {market_pcts.max():.1f}%]")
print(f"t-test vs 50%: t={t_stat:.3f}, p={p_val:.4f}")

# Chart: histogram of per-market YES buy %
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(market_pcts, bins=20, color="#4ECDC4", edgecolor="white", alpha=0.8)
ax.axvline(50, color="#e74c3c", ls="--", lw=2, label="50% (no bias)")
ax.axvline(market_pcts.mean(), color="#2c3e50", ls="-", lw=2, label=f"Mean: {market_pcts.mean():.1f}%")
ax.axvline(55, color="#f39c12", ls=":", lw=2, label="55% (pre-registered threshold)")
ax.set_xlabel("YES buy % per market")
ax.set_ylabel("Number of markets")
ax.set_title("H1: Distribution of YES Buy Proportion Across 88 Markets")
ax.legend()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "03-h1-yes-buy-distribution.png"), dpi=150, bbox_inches="tight")

### H1 Result: Not confirmed

YES buy proportion is 51.3% (95% CI: 50.7–52.0%). Statistically significant (p=0.0001) but well below the pre-registered 55% threshold. The effect is 1.3 percentage points above parity.

At the market level, the histogram shows most markets cluster tightly around 50% (std dev ~4pp, range 43–65%). The order book enforces complementary pricing (YES + NO ≈ $1) — when YES buying pushes the price up, NO becomes cheaper, attracting NO buyers. This mechanism structurally limits how far trade-count ratios can deviate from 50%.

**Bottom line:** There may be a small YES lean, but trade counts are too constrained by market microstructure to detect behavioral bias, even if it exists. This metric is insensitive to the question we're asking.

---
## Context: The BUY/SELL Ratio Is Not Bias — It's Position Accumulation

Our V1 analysis found ~65% of YES token trades were BUYs, which we initially interpreted as "YES buying bias." With both token types now in the data, we can test whether this pattern is YES-specific or applies equally to NO tokens.

In [ ]:
yes_trades = df[df["token_type"] == "YES"]
no_trades = df[df["token_type"] == "NO"]

yes_buy_rate = (yes_trades["side"] == "BUY").mean() * 100
no_buy_rate = (no_trades["side"] == "BUY").mean() * 100

print(f"YES token buy rate: {yes_buy_rate:.1f}%")
print(f"NO token buy rate:  {no_buy_rate:.1f}%")
print(f"Difference: {yes_buy_rate - no_buy_rate:+.1f}pp")

# Chart
fig, ax = plt.subplots(figsize=(7, 4))
x = [0, 1]
bars = ax.bar(x, [yes_buy_rate, no_buy_rate], color=["#4ECDC4", "#C44D58"], width=0.5)
ax.set_xticks(x)
ax.set_xticklabels(["YES tokens", "NO tokens"])
ax.set_ylabel("BUY trades (%)")
ax.set_title("BUY/SELL Ratio by Token Type")
ax.axhline(50, color="grey", ls="--", alpha=0.5)
ax.set_ylim(0, 100)
for bar, val in zip(bars, [yes_buy_rate, no_buy_rate]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, f"{val:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "04-buysell-ratio.png"), dpi=150, bbox_inches="tight")

### Interpretation

The ~77% buy rate is nearly identical on both token types. This is not YES bias — it's position accumulation. Polymarket reports ~60% of traders hold positions to resolution, meaning every held position generates a BUY trade with no matching SELL.

**V1 correction:** The earlier analysis only examined YES token trades and misattributed this structural BUY/SELL imbalance as YES-specific preference. It was an artifact of incomplete data, not a real finding.

---
## H2: Are YES Tokens Overpriced?

Trade counts can't detect bias (H1 showed this). But **prices** might. Deleep et al. (2026) found YES tokens are systematically overpriced relative to actual outcomes across 5,456 markets. We test this on our 88 single-market events.

**Method:** For each market, compute the volume-weighted average price (VWAP) of YES token buys. Group markets by VWAP bucket. Compare average VWAP to actual YES resolution rate. If well-calibrated, VWAP ≈ resolution rate. If YES is overpriced, VWAP > resolution rate.

**Pre-registered threshold:** Average overpricing > 5pp across buckets.

**Important caveat:** 88 markets is small for calibration. Deleep et al. used 5,456.

In [ ]:
# Calibration
cal = compute_calibration(df, n_buckets=5)

print("=== Calibration: YES VWAP vs Resolution Rate ===")
print(f"{'Bucket':>10s}  {'Markets':>8s}  {'Avg VWAP':>9s}  {'Resolution':>11s}  {'Overpricing':>12s}")
for _, row in cal.iterrows():
    flag = " *" if row["n_markets"] < 10 else "  "
    print(f"{row['bucket']:>10s}  {row['n_markets']:>7d}{flag}  {row['avg_vwap']*100:>8.1f}%  {row['resolution_rate']*100:>10.1f}%  {row['overpricing']*100:>+11.1f}pp")
print("\n* = fewer than 10 markets (low confidence)")

# Chart: calibration plot — VWAP vs resolution rate
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot([0, 100], [0, 100], "k--", alpha=0.3, label="Perfect calibration")
ax.scatter(
    cal["avg_vwap"] * 100, cal["resolution_rate"] * 100,
    s=cal["n_markets"] * 15, c="#4ECDC4", edgecolor="#2c3e50", zorder=5,
)
for _, row in cal.iterrows():
    ax.annotate(
        f"n={row['n_markets']}",
        (row["avg_vwap"]*100, row["resolution_rate"]*100),
        textcoords="offset points", xytext=(10, -5), fontsize=9,
    )

ax.set_xlabel("Average YES VWAP (%)")
ax.set_ylabel("Actual YES Resolution Rate (%)")
ax.set_title("H2: Calibration — Are YES Tokens Overpriced?")
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "05-h2-calibration.png"), dpi=150, bbox_inches="tight")

In [ ]:
# Buyer P&L — the complement to calibration
pnl = compute_buyer_pnl(df)

print("=== Buyer P&L at resolution ===")
print(f"YES buyers: mean P&L = {pnl['yes_buyer_mean_pnl']:+.3f} per trade (n={pnl['yes_buyer_n_trades']:,})")
print(f"NO buyers:  mean P&L = {pnl['no_buyer_mean_pnl']:+.3f} per trade (n={pnl['no_buyer_n_trades']:,})")
print(f"Gap: {pnl['pnl_gap']:+.3f}")

# P&L by price bucket
buys = df[df["side"] == "BUY"].copy()
buys = buys[buys["resolved_yes"].notna()]
buys.loc[buys["token_type"] == "YES", "pnl"] = buys.loc[buys["token_type"] == "YES", "resolved_yes"].astype(float) - buys.loc[buys["token_type"] == "YES", "price"]
buys.loc[buys["token_type"] == "NO", "pnl"] = (~buys.loc[buys["token_type"] == "NO", "resolved_yes"]).astype(float) - buys.loc[buys["token_type"] == "NO", "price"]

# Chart: P&L by token type and price bucket
fig, ax = plt.subplots(figsize=(10, 5))
price_labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]
price_bins = [(0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]

yes_pnls = []
no_pnls = []
for lo, hi in price_bins:
    yb = buys[(buys["token_type"] == "YES") & (buys["price"] >= lo) & (buys["price"] < hi)]
    nb = buys[(buys["token_type"] == "NO") & (buys["price"] >= lo) & (buys["price"] < hi)]
    yes_pnls.append(yb["pnl"].mean() if len(yb) >= 10 else np.nan)
    no_pnls.append(nb["pnl"].mean() if len(nb) >= 10 else np.nan)

x = np.arange(len(price_labels))
w = 0.35
ax.bar(x - w/2, yes_pnls, w, label="YES buyers", color="#4ECDC4")
ax.bar(x + w/2, no_pnls, w, label="NO buyers", color="#C44D58")
ax.axhline(0, color="black", lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(price_labels)
ax.set_xlabel("Token price at purchase")
ax.set_ylabel("Mean P&L per trade")
ax.set_title("H2: Buyer P&L by Price Bucket")
ax.legend()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "06-h2-pnl-by-price.png"), dpi=150, bbox_inches="tight")

### H2 Result: Not confirmed — our sample shows underpricing, not overpricing

In our 88-market sample, YES tokens appear **underpriced**: YES buyers earn +0.149 per trade on average, while NO buyers lose -0.158. The calibration chart shows most points above the diagonal, meaning markets resolve YES more often than their price implied.

**This contradicts Deleep et al.**, who found systematic YES overpricing across 5,456 markets. Possible explanations:

1. **Sample size.** 88 markets with 3–26 per calibration bucket is small. A few surprising resolutions can swing an entire bucket.
2. **Selection bias.** Our top-500-by-volume filter may over-represent "surprising" outcomes that resolved against expectations, inflating resolution rates in low-VWAP buckets.
3. **VWAP temporal bias.** Our 200-trade cap samples early-life trading. Early prices run ~2pp lower than late prices (not significant, p=0.076) — this could partially explain underpricing.
4. **Population difference.** We analyze single-market events only. Deleep et al. include multi-market events (elections, crypto, sports) with different dynamics.

**Bottom line:** We cannot confirm YES overpricing in our sample, but we also cannot reject Deleep et al.'s finding. Their study is 62x larger. This remains an open question requiring more data.

---
## H3: Do Small and Large Traders Differ?

This was the most interesting signal in our data. By trade count, YES/NO buying is nearly 50/50 (H1). But by dollar volume, YES buys account for only ~25% — meaning NO buyers deploy roughly 3x more capital. This suggests trade size correlates with token preference.

**Pre-registered threshold:** YES buy % differs by > 10pp between smallest and largest trade size buckets.

In [ ]:
# Raw trade-size gradient — YES and NO buys side by side
buys_all = df[df["side"] == "BUY"].copy()

size_bins = [(0, 20, "<$20"), (20, 100, "$20–100"), (100, 500, "$100–500"), (500, float("inf"), "$500+")]
rows = []
for lo, hi, label in size_bins:
    bucket = buys_all[(buys_all["usdc"] >= lo) & (buys_all["usdc"] < hi)]
    if len(bucket) < 10:
        continue
    yes_n = int((bucket["token_type"] == "YES").sum())
    no_n = len(bucket) - yes_n
    total = len(bucket)
    rows.append({"label": label, "trades": total, "yes_buys": yes_n, "no_buys": no_n,
                  "yes_pct": yes_n / total * 100, "no_pct": no_n / total * 100,
                  "avg_usdc": bucket["usdc"].mean()})

size_df = pd.DataFrame(rows)
print("=== YES vs NO buy preference by trade size ===")
print(f"{'Size':>12s}  {'Trades':>7s}  {'YES %':>7s}  {'NO %':>7s}")
for _, r in size_df.iterrows():
    print(f"  {r['label']:>12s}  {r['trades']:>7,}  {r['yes_pct']:.1f}%  {r['no_pct']:.1f}%")

# Correlation
is_yes = (buys_all["token_type"] == "YES").astype(int)
corr, p = stats.pointbiserialr(is_yes, buys_all["usdc"])
print(f"\nPoint-biserial correlation: r={corr:.4f}, p={p:.2e}")

# Chart: grouped bars — YES% and NO% by trade size
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(size_df))
w = 0.35
bars_yes = ax.bar(x - w/2, size_df["yes_pct"], w, label="YES buys", color="#4ECDC4")
bars_no = ax.bar(x + w/2, size_df["no_pct"], w, label="NO buys", color="#C44D58")
ax.axhline(50, color="grey", ls="--", alpha=0.5, label="50% (no preference)")
ax.set_xticks(x)
ax.set_xticklabels(size_df["label"])
ax.set_xlabel("Trade size")
ax.set_ylabel("Share of buys (%)")
ax.set_title("H3: YES vs NO Token Preference by Trade Size")
ax.set_ylim(0, 70)
for bar, val, n in zip(bars_yes, size_df["yes_pct"], size_df["trades"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8, f"{val:.1f}%", ha="center", fontsize=9)
for bar, val in zip(bars_no, size_df["no_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.8, f"{val:.1f}%", ha="center", fontsize=9)
# Add trade counts below x labels
for i, n in enumerate(size_df["trades"]):
    ax.text(i, -4.5, f"n={n:,}", ha="center", fontsize=8, color="grey")
ax.legend()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "07-h3-size-gradient.png"), dpi=150, bbox_inches="tight")

### The gradient is 19pp — well above the 10pp threshold

Small trades (<$20) are 56% YES. Large trades ($500+) are 37% YES. The gap is 19 percentage points and statistically significant (p < 0.001).

But this aggregate gradient could be a composition effect — maybe cheap markets attract both small trades and more YES buying. We need to test whether it holds (a) within individual markets and (b) when controlling for price.

In [ ]:
# Within-market test: split each market's trades at median size
buys_wm = df[df["side"] == "BUY"].copy()
small_yes = []
large_yes = []

for q in buys_wm["question"].unique():
    m = buys_wm[buys_wm["question"] == q]
    if len(m) < 40:
        continue
    med = m["usdc"].median()
    small = m[m["usdc"] <= med]
    large = m[m["usdc"] > med]
    if len(small) < 10 or len(large) < 10:
        continue
    small_yes.append((small["token_type"] == "YES").mean() * 100)
    large_yes.append((large["token_type"] == "YES").mean() * 100)

small_yes = np.array(small_yes)
large_yes = np.array(large_yes)
diff = small_yes - large_yes
t, p = stats.ttest_rel(small_yes, large_yes)

print(f"Markets analyzed: {len(small_yes)}")
print(f"Mean YES% — small trades: {small_yes.mean():.1f}%, large trades: {large_yes.mean():.1f}%")
print(f"Mean gap: {diff.mean():.1f}pp, paired t-test: t={t:.2f}, p={p:.2e}")
print(f"Markets where small > large: {(diff > 0).sum()}/{len(diff)} ({(diff>0).mean()*100:.0f}%)")

# Chart: paired comparison — scatter of small vs large YES% per market
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(large_yes, small_yes, alpha=0.4, s=30, c="#556270")
ax.plot([35, 75], [35, 75], "k--", alpha=0.3, label="Equal (no gradient)")
ax.set_xlabel("Large trades: YES buy %")
ax.set_ylabel("Small trades: YES buy %")
ax.set_title(f"Within-Market Size Gradient ({len(small_yes)} markets)")
ax.text(0.05, 0.95, f"Mean gap: {diff.mean():.1f}pp\np = {p:.2e}\n{(diff>0).sum()}/{len(diff)} markets above diagonal",
        transform=ax.transAxes, va="top", fontsize=10,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
ax.legend()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "08-h3-within-market.png"), dpi=150, bbox_inches="tight")

### The gradient holds within individual markets

In 63 of 88 markets, small trades lean more YES than large trades (mean gap: 12.8pp, p < 0.0001). This rules out a simple "cheap markets attract both small trades and YES buying" composition story — the gradient exists *inside* the same market.

But this still doesn't tell us **why**. Is it behavioral (small traders genuinely prefer YES)? Or structural (small trades happen at price points where YES is the cheaper token)? We need to control for price.

In [ ]:
# Price-controlled analysis: does the gradient survive within price buckets?
buys_pc = df[df["side"] == "BUY"].copy()
buys_pc["price_bucket"] = pd.cut(
    buys_pc["price"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["0–20%", "20–40%", "40–60%", "60–80%", "80–100%"]
)

# Build the table
price_labels = ["0–20%", "20–40%", "40–60%", "60–80%", "80–100%"]
size_cats = [("<$20", 0, 20), ("$20–100", 20, 100), (">$100", 100, float("inf"))]

table_data = {}
for pb in price_labels:
    pb_buys = buys_pc[buys_pc["price_bucket"] == pb]
    row = {}
    for slabel, slo, shi in size_cats:
        subset = pb_buys[(pb_buys["usdc"] >= slo) & (pb_buys["usdc"] < shi)]
        if len(subset) >= 10:
            row[slabel] = {"pct": (subset["token_type"] == "YES").mean() * 100, "n": len(subset)}
        else:
            row[slabel] = {"pct": np.nan, "n": len(subset)}
    table_data[pb] = row

print("=== YES% by trade size, controlling for price level ===")
print(f"{'Price':>10s}  {'<$20':>10s}  {'$20–100':>10s}  {'>$100':>10s}")
for pb in price_labels:
    parts = []
    for slabel, _, _ in size_cats:
        d = table_data[pb][slabel]
        parts.append(f"{d['pct']:.1f}%" if not np.isnan(d['pct']) else "n/a")
    print(f"{pb:>10s}  {parts[0]:>10s}  {parts[1]:>10s}  {parts[2]:>10s}")

# Formal within-bucket correlation tests
print("\n=== Within-price-bucket correlation (log trade size ~ YES token) ===")
buys_pc["is_yes"] = (buys_pc["token_type"] == "YES").astype(int)
buys_pc["log_usdc"] = np.log1p(buys_pc["usdc"])
for pb in price_labels:
    pb_data = buys_pc[buys_pc["price_bucket"] == pb]
    if len(pb_data) < 50:
        continue
    corr, p = stats.pointbiserialr(pb_data["is_yes"], pb_data["log_usdc"])
    sig = "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"  {pb}: r={corr:+.4f}, p={p:.4f} {sig}  (n={len(pb_data):,})")

# Chart: heatmap-style grouped bar chart
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(price_labels))
w = 0.25
for i, (slabel, _, _) in enumerate(size_cats):
    vals = [table_data[pb][slabel]["pct"] for pb in price_labels]
    ax.bar(x + (i-1)*w, vals, w, label=slabel, alpha=0.85)

ax.axhline(50, color="grey", ls="--", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(price_labels)
ax.set_xlabel("Token price level")
ax.set_ylabel("YES buy %")
ax.set_title("H3: YES Preference by Trade Size, Controlling for Price")
ax.legend(title="Trade size")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "09-h3-price-controlled.png"), dpi=150, bbox_inches="tight")

### Price control: the gradient is mostly — but not entirely — a composition effect

The grouped bar chart tells the story visually. Within each price bucket, the three trade-size groups show similar YES percentages. The dominant pattern is **price level**, not trade size: at low prices everyone buys YES (it's the cheap token); at high prices everyone buys NO.

But the formal within-bucket correlations reveal it's not perfectly clean:

| Price bucket | Correlation (size ~ YES) | Direction |
|---|---|---|
| 0–20% | r=+0.03, p=0.06 | Near-zero, not significant |
| 20–40% | r=−0.04, p=0.006 | Larger trades slightly less YES ✓ |
| 40–60% | r=+0.01, p=0.64 | No relationship |
| 60–80% | r=+0.07, p<0.001 | Larger trades **more** YES (opposite!) |
| 80–100% | r=−0.07, p<0.001 | Larger trades less YES ✓ |

At 60–80% prices, larger traders actually prefer YES more, not less. The direction is inconsistent across price levels. This means the aggregate 19pp gradient is primarily explained by **where different-sized trades land on the price spectrum**, but there may be residual effects that don't follow a simple narrative.

**Honest assessment:** The price composition explanation accounts for most of the gradient. The within-bucket residuals are real but small and contradictory in direction — they don't support a clean "small traders are biased" story.

In [ ]:
# Where do different-sized trades land on the price spectrum?
buys_dist = df[df["side"] == "BUY"].copy()
buys_dist["price_bucket"] = pd.cut(
    buys_dist["price"], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["0–20%", "20–40%", "40–60%", "60–80%", "80–100%"]
)

fig, ax = plt.subplots(figsize=(10, 5))
size_buckets = [("<$20", 0, 20), ("$20–100", 20, 100), ("$100–500", 100, 500), ("$500+", 500, float("inf"))]
x = np.arange(5)
w = 0.2

for i, (slabel, slo, shi) in enumerate(size_buckets):
    subset = buys_dist[(buys_dist["usdc"] >= slo) & (buys_dist["usdc"] < shi)]
    total = len(subset)
    pcts = []
    for pb in ["0–20%", "20–40%", "40–60%", "60–80%", "80–100%"]:
        n = len(subset[subset["price_bucket"] == pb])
        pcts.append(n / total * 100)
    ax.bar(x + (i - 1.5) * w, pcts, w, label=slabel, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(["0–20%", "20–40%", "40–60%", "60–80%", "80–100%"])
ax.set_xlabel("Token price level")
ax.set_ylabel("% of trades in this size bucket")
ax.set_title("Where Different-Sized Trades Land on the Price Spectrum")
ax.legend(title="Trade size")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "10-h3-size-price-distribution.png"), dpi=150, bbox_inches="tight")

### The mechanism: trade size determines price access

The distribution chart shows why the gradient exists. Small trades (<$20) are spread fairly evenly across prices but skew toward the cheap end (26% at 0–20%). Large trades ($500+) concentrate at the expensive end (37% at 80–100%, only 6% at 0–20%).

At low prices, YES is the cheap token (a YES token at 10% costs $0.10). At high prices, NO is the cheap token (NO at 80% YES price costs $0.20). Traders of all sizes appear to buy whichever token is cheaper at their price point. The "small traders prefer YES" narrative is actually "small trades happen where YES is cheap."

**H3 verdict:** The 19pp raw gradient exceeds the pre-registered 10pp threshold, so H3 is technically confirmed. But the mechanism is price composition, not behavioral preference. The finding is informative about market microstructure — it tells us how capital of different sizes accesses different parts of the price spectrum — but it does not support a story about small-trader irrationality or YES bias.

---
## Question Framing and the Longshot Confound

Our path to 88 single-market events revealed something we didn't expect. By filtering to single-outcome markets we ended up with a population that is overwhelmingly longshot-by-design: the median YES price is 34%, and 41 of 88 markets have an average YES price below 30%. Only 4 markets have YES price above 70%.

This isn't a sampling accident — it reflects how Polymarket frames questions. The platform's most popular single-market events are "Will [unlikely/dramatic thing] happen?" questions: Trump resigning, Bitcoin hitting $250k, aliens confirmed. The editorial incentive is clear: dramatic longshot questions generate volume and engagement.

This creates a structural confound: in most markets, YES = the longshot token. Any longshot preference (well-documented since Griffith 1949) will therefore appear as YES preference. But is it really about "YES"? Or is it about cheapness?

In [ ]:
# Market price distribution — how our sample skews
yes_only = df[df["token_type"] == "YES"]
market_avg_price = yes_only.groupby("question")["price"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: histogram of market average YES prices
axes[0].hist(market_avg_price, bins=20, color="#556270", edgecolor="white")
axes[0].axvline(market_avg_price.median(), color="#e74c3c", ls="--", lw=2,
                label=f"Median: {market_avg_price.median():.0%}")
axes[0].axvline(0.5, color="#f39c12", ls=":", lw=2, label="50%")
axes[0].set_xlabel("Average YES price per market")
axes[0].set_ylabel("Number of markets")
axes[0].set_title("Our 88 markets: YES price distribution")
axes[0].legend()

# Right: in high-YES-price markets, do traders still prefer YES?
# Compare YES buy % in low-price vs high-price markets
low_price_markets = market_avg_price[market_avg_price < 0.30].index
mid_price_markets = market_avg_price[(market_avg_price >= 0.30) & (market_avg_price <= 0.60)].index
high_price_markets = market_avg_price[market_avg_price > 0.60].index

buys_framing = df[df["side"] == "BUY"]
categories = []
for label, mkts in [("YES < 30%\n(YES is cheap)", low_price_markets),
                     ("YES 30–60%\n(both similar)", mid_price_markets),
                     ("YES > 60%\n(NO is cheap)", high_price_markets)]:
    b = buys_framing[buys_framing["question"].isin(mkts)]
    yes_pct = (b["token_type"] == "YES").mean() * 100
    categories.append({"label": label, "yes_pct": yes_pct, "n": len(b)})

cat_df = pd.DataFrame(categories)
colors = ["#4ECDC4", "#556270", "#C44D58"]
bars = axes[1].bar(cat_df["label"], cat_df["yes_pct"], color=colors)
axes[1].axhline(50, color="grey", ls="--", alpha=0.5)
axes[1].set_ylabel("YES buy %")
axes[1].set_title("Token preference follows price, not label")
axes[1].set_ylim(0, 80)
for bar, row in zip(bars, cat_df.itertuples()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f"{row.yes_pct:.1f}%\n(n={row.n:,})", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "14-question-framing.png"), dpi=150, bbox_inches="tight")

In [ ]:
# The symmetry test: at equivalent token costs, do traders prefer YES or NO?
buys_sym = df[df["side"] == "BUY"].copy()

# Bin by what the trader PAID per token (regardless of YES/NO)
buys_sym["cost_bucket"] = pd.cut(buys_sym["price"],
    bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    labels=["1–10c", "10–20c", "20–30c", "30–40c", "40–50c",
            "50–60c", "60–70c", "70–80c", "80–90c", "90–100c"])

sym_rows = []
for cb in buys_sym["cost_bucket"].cat.categories:
    b = buys_sym[buys_sym["cost_bucket"] == cb]
    if len(b) < 20:
        continue
    yes_n = int((b["token_type"] == "YES").sum())
    sym_rows.append({"bucket": cb, "total": len(b), "yes": yes_n,
                     "no": len(b) - yes_n, "yes_pct": yes_n / len(b) * 100})

sym_df = pd.DataFrame(sym_rows)

print("=== YES share of buys by token cost ===")
print("If longshot bias is symmetric, cheap tokens should be ~50/50 YES/NO")
print("If YES-specific, cheap tokens should skew YES even vs cheap NO")
print()
print(f"{'Cost':>10s}  {'Total':>7s}  {'YES':>7s}  {'NO':>7s}  {'YES%':>7s}")
for _, r in sym_df.iterrows():
    print(f"{r['bucket']:>10s}  {r['total']:>7,}  {r['yes']:>7,}  {r['no']:>7,}  {r['yes_pct']:>6.1f}%")

# Chart: YES% by token cost — showing the price-driven curve
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(sym_df))
ax.bar(x, sym_df["yes_pct"], color=["#4ECDC4" if p > 55 else "#C44D58" if p < 45 else "#556270"
                                     for p in sym_df["yes_pct"]], edgecolor="white")
ax.axhline(50, color="grey", ls="--", alpha=0.5, label="50%")
ax.set_xticks(x)
ax.set_xticklabels(sym_df["bucket"], rotation=45, ha="right")
ax.set_xlabel("Token cost (what the trader paid per token)")
ax.set_ylabel("YES share of buys (%)")
ax.set_title("Longshot Symmetry: Traders Buy Whatever Is Cheap")
ax.set_ylim(0, 100)

# Annotate
ax.annotate("Cheap YES tokens →\ntraders buy YES", xy=(1, 90), fontsize=10, color="#4ECDC4", fontweight="bold")
ax.annotate("← Cheap NO tokens\ntraders buy NO", xy=(7, 15), fontsize=10, color="#C44D58", fontweight="bold")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "15-longshot-symmetry.png"), dpi=150, bbox_inches="tight")

### The framing confound

The data tells a clear story. When we bin all buys by token cost (what the trader paid per token, regardless of YES/NO label):

- At 1–20c: ~90% of buys are YES tokens. But these are markets where YES is the cheap longshot.
- At 80–100c: ~90% of buys are NO tokens. These are the same markets — NO is now the cheap longshot from the other side.
- At 40–60c: it's 50/50. Neither token is cheap.

In the few markets where YES is the favourite (price > 60%), traders buy cheap NO at 50–55%, not expensive YES. The preference follows price, not the label.

**What this means for "YES bias" as a concept:**

Polymarket's question framing creates a structural alignment between YES and longshot. Most single-market questions are "Will [unlikely thing] happen?" — meaning YES is cheap in the majority of markets. Any longshot preference (people buying cheap tokens hoping for outsized payoffs) will therefore register as YES preference in aggregate data.

To truly isolate YES-specific bias from longshot bias, you would need either:
1. A platform that randomises which outcome gets the YES label
2. A large sample of "inverted" markets where YES is the favourite (our sample has only 4)
3. The same question framed both ways on different platforms

Our data suggests that what has been called "YES bias" may be substantially — perhaps entirely — a longshot bias channelled through editorial question framing. Traders don't prefer YES. They prefer cheap. Polymarket's question design makes "cheap" and "YES" synonymous in most markets.

**This has implications beyond our study.** If platforms optimise question framing for volume (dramatic, unlikely events generate more engagement), they are also shaping which token becomes the longshot — and therefore which token appears "biased." The editorial layer is not neutral; it's load-bearing in how bias shows up in the data.

A simple transparency measure: platforms could randomise or alternate which outcome is labeled YES. This would decorrelate the YES label from the longshot position and allow researchers to separate genuine affirmative preference from longshot preference. Until then, "YES bias" and "longshot bias" cannot be cleanly distinguished.

---
## Tangential Findings

These findings emerged during analysis but were not part of the original hypotheses. They're included for completeness and as leads for future work, not as confirmed results.

In [ ]:
# P&L by trade size — do small traders make money?
buys_pnl = df[df["side"] == "BUY"].copy()
buys_pnl = buys_pnl[buys_pnl["resolved_yes"].notna()]
buys_pnl.loc[buys_pnl["token_type"] == "YES", "pnl"] = buys_pnl.loc[buys_pnl["token_type"] == "YES", "resolved_yes"].astype(float) - buys_pnl.loc[buys_pnl["token_type"] == "YES", "price"]
buys_pnl.loc[buys_pnl["token_type"] == "NO", "pnl"] = (~buys_pnl.loc[buys_pnl["token_type"] == "NO", "resolved_yes"]).astype(float) - buys_pnl.loc[buys_pnl["token_type"] == "NO", "price"]

print("=== P&L by trade size ===")
print(f"{'Size':>12s}  {'Mean P&L':>9s}  {'YES P&L':>9s}  {'NO P&L':>9s}  {'n':>7s}")
size_labels = []
all_pnl = []
yes_pnl_list = []
no_pnl_list = []
for lo, hi, label in [(0, 20, "<$20"), (20, 100, "$20–100"), (100, 500, "$100–500"), (500, float("inf"), "$500+")]:
    b = buys_pnl[(buys_pnl["usdc"] >= lo) & (buys_pnl["usdc"] < hi)]
    if len(b) < 20:
        continue
    yes_b = b[b["token_type"] == "YES"]
    no_b = b[b["token_type"] == "NO"]
    print(f"{label:>12s}  {b['pnl'].mean():>+8.3f}  {yes_b['pnl'].mean():>+8.3f}  {no_b['pnl'].mean():>+8.3f}  {len(b):>7,}")
    size_labels.append(label)
    all_pnl.append(b["pnl"].mean())
    yes_pnl_list.append(yes_b["pnl"].mean())
    no_pnl_list.append(no_b["pnl"].mean())

# Chart
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(size_labels))
w = 0.25
ax.bar(x - w, yes_pnl_list, w, label="YES buyers", color="#4ECDC4")
ax.bar(x, no_pnl_list, w, label="NO buyers", color="#C44D58")
ax.bar(x + w, all_pnl, w, label="All buyers", color="#556270")
ax.axhline(0, color="black", lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels(size_labels)
ax.set_xlabel("Trade size")
ax.set_ylabel("Mean P&L per trade")
ax.set_title("Tangential: P&L by Trade Size and Token Type")
ax.legend()
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "11-tangential-pnl-by-size.png"), dpi=150, bbox_inches="tight")

### T1: Small trades are the only profitable bucket

Small trades (<$20) earn +$0.024 per trade on average. Every larger bucket is negative. YES buyers are profitable at every size, but their edge shrinks monotonically with trade size. NO buyers lose money at every size.

**What this could mean:** Small traders may capture edge by buying cheap YES tokens on events that resolve YES at rates above their price. Or it could be survivorship bias in our 88-market sample (we know our calibration shows underpricing, which mechanically benefits YES buyers).

**What further examination would need:** Replicate on a larger sample (500+ markets). Control for market vintage (are small trades concentrated in markets that happened to resolve favorably?). Test whether the P&L difference is statistically significant or within noise for 88 markets.

In [ ]:
# Wallet-level analysis
buys_w = df[df["side"] == "BUY"].copy()
wallet_stats = buys_w.groupby("maker").agg(
    n_trades=("timestamp", "count"),
    mean_size=("usdc", "mean"),
    total_vol=("usdc", "sum"),
    yes_pct=("token_type", lambda x: (x == "YES").mean() * 100),
).reset_index()

active = wallet_stats[wallet_stats["n_trades"] >= 5]
small_w = active[active["mean_size"] < active["mean_size"].median()]
large_w = active[active["mean_size"] >= active["mean_size"].median()]

t, p = stats.ttest_ind(small_w["yes_pct"], large_w["yes_pct"])

print(f"Wallets with 5+ trades: {len(active):,}")
print(f"Small wallets (below median avg size): n={len(small_w)}, mean YES%={small_w['yes_pct'].mean():.1f}%")
print(f"Large wallets (above median avg size): n={len(large_w)}, mean YES%={large_w['yes_pct'].mean():.1f}%")
print(f"Difference: {small_w['yes_pct'].mean() - large_w['yes_pct'].mean():+.1f}pp, p={p:.4f}")

# Chart: distribution of YES% by wallet size
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

ax1.hist(small_w["yes_pct"], bins=20, color="#4ECDC4", edgecolor="white", alpha=0.8)
ax1.axvline(50, color="#e74c3c", ls="--")
ax1.axvline(small_w["yes_pct"].mean(), color="#2c3e50", ls="-", lw=2)
ax1.set_xlabel("YES buy %")
ax1.set_ylabel("Number of wallets")
ax1.set_title(f"Small wallets (n={len(small_w)}, mean={small_w['yes_pct'].mean():.1f}%)")

ax2.hist(large_w["yes_pct"], bins=20, color="#C44D58", edgecolor="white", alpha=0.8)
ax2.axvline(50, color="#e74c3c", ls="--")
ax2.axvline(large_w["yes_pct"].mean(), color="#2c3e50", ls="-", lw=2)
ax2.set_xlabel("YES buy %")
ax2.set_title(f"Large wallets (n={len(large_w)}, mean={large_w['yes_pct'].mean():.1f}%)")

plt.suptitle("Tangential: Wallet-Level YES Preference by Trader Size", y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "12-tangential-wallet-distribution.png"), dpi=150, bbox_inches="tight")

### T2: Small wallets lean YES, large wallets lean NO

The wallet-level split confirms the trade-level pattern: small-wallet makers average ~55% YES, large-wallet makers ~42% YES. The difference is significant (p < 0.001).

**What this could mean:** Different wallet sizes access different price ranges (consistent with H3 findings). Or large wallets may include market makers who systematically provide liquidity on the NO side. Or institutional/whale capital may concentrate on high-probability events where NO is the cheap token.

**What further examination would need:** Track wallet behavior across multiple markets to separate "wallet preference" from "market selection." Identify market maker wallets (high trade counts, balanced buy/sell) vs. directional traders.

---
## Robustness Check: Maker vs Taker

Our trade data labels BUY/SELL from the **maker's** perspective (the limit order placer). A reasonable concern: if professional market makers dominate the maker side, they'd provide two-sided liquidity and push YES/NO ratios toward 50/50 by construction — making our H1 result (51.3%) an artifact.

We test this by flipping the lens. When a maker SELLs YES tokens, the **taker** (the one who crossed the spread) is BUYing YES tokens. Takers express directional conviction — they pay the spread to get into a position. If YES bias exists, taker-side should show it.

In [ ]:
# Taker-side = maker SELL trades (taker crossed the spread to buy tokens)
taker_buys = df[df["side"] == "SELL"].copy()
taker_buys_yes = int((taker_buys["token_type"] == "YES").sum())
taker_buys_total = len(taker_buys)

binom_taker = stats.binomtest(taker_buys_yes, taker_buys_total, 0.5)

print("=== Maker-side vs Taker-side YES preference ===")
maker_buys_total = len(df[df["side"] == "BUY"])
maker_buys_yes = int((df[df["side"] == "BUY"]["token_type"] == "YES").sum())
print(f"Maker BUYs targeting YES: {maker_buys_yes/maker_buys_total*100:.1f}% (n={maker_buys_total:,})")
print(f"Taker BUYs targeting YES: {taker_buys_yes/taker_buys_total*100:.1f}% (n={taker_buys_total:,}, p={binom_taker.pvalue:.4f})")
print()

# Are makers dominated by market-making bots?
maker_profile = df.groupby("maker").agg(
    n=("side", "count"),
    buy_pct=("side", lambda x: (x == "BUY").mean() * 100),
).reset_index()
active = maker_profile[maker_profile["n"] >= 10]

balanced = ((active["buy_pct"] > 30) & (active["buy_pct"] < 70)).sum()
buy_heavy = (active["buy_pct"] > 70).sum()

print(f"=== Maker wallet profiles (wallets with 10+ trades) ===")
print(f"Total: {len(active)}")
print(f"Balanced (30-70% buy, likely market makers): {balanced} ({balanced/len(active)*100:.0f}%)")
print(f"Buy-heavy (>70% buy, directional traders): {buy_heavy} ({buy_heavy/len(active)*100:.0f}%)")
print()

# Taker-side trade size gradient
print("=== Taker-side trade size gradient ===")
print(f"{'Size':>12s}  {'Maker YES%':>11s}  {'Taker YES%':>11s}  {'Taker n':>8s}")
maker_buys = df[df["side"] == "BUY"]
for lo, hi, label in [(0, 20, "<$20"), (20, 100, "$20-100"), (100, 500, "$100-500"), (500, float("inf"), "$500+")]:
    mb = maker_buys[(maker_buys["usdc"] >= lo) & (maker_buys["usdc"] < hi)]
    tb = taker_buys[(taker_buys["usdc"] >= lo) & (taker_buys["usdc"] < hi)]
    if len(tb) < 10:
        continue
    m_pct = (mb["token_type"] == "YES").mean() * 100
    t_pct = (tb["token_type"] == "YES").mean() * 100
    print(f"{label:>12s}  {m_pct:>10.1f}%  {t_pct:>10.1f}%  {len(tb):>8,}")

# Chart: side-by-side maker vs taker
fig, ax = plt.subplots(figsize=(8, 5))
size_labels = ["<$20", "$20-100", "$100-500", "$500+"]
size_bins = [(0, 20), (20, 100), (100, 500), (500, float("inf"))]

maker_pcts = []
taker_pcts = []
for lo, hi in size_bins:
    mb = maker_buys[(maker_buys["usdc"] >= lo) & (maker_buys["usdc"] < hi)]
    tb = taker_buys[(taker_buys["usdc"] >= lo) & (taker_buys["usdc"] < hi)]
    maker_pcts.append((mb["token_type"] == "YES").mean() * 100 if len(mb) >= 10 else np.nan)
    taker_pcts.append((tb["token_type"] == "YES").mean() * 100 if len(tb) >= 10 else np.nan)

x = np.arange(len(size_labels))
w = 0.35
ax.bar(x - w/2, maker_pcts, w, label="Maker-side (limit orders)", color="#4ECDC4")
ax.bar(x + w/2, taker_pcts, w, label="Taker-side (market orders)", color="#556270")
ax.axhline(50, color="grey", ls="--", alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(size_labels)
ax.set_xlabel("Trade size")
ax.set_ylabel("YES buy %")
ax.set_title("Robustness: Maker vs Taker YES Preference by Trade Size")
ax.legend()
ax.set_ylim(30, 65)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "13-robustness-maker-taker.png"), dpi=150, bbox_inches="tight")

### Robustness result

The taker-side tells the same story as the maker-side:

- **H1:** Taker YES buy % is 51.5% (vs maker 51.3%). The ~50/50 result is not a market-maker artifact — it's the order book's complementary pricing at work.
- **H3:** The trade-size gradient holds on both sides (taker: 54.6% to 39.6%, maker: 56.1% to 37.0%). Slightly compressed on the taker side but same direction and magnitude.
- **Maker population:** Only 35% of active maker wallets look like two-sided market makers. 57% are buy-heavy directional traders who happen to use limit orders.

The maker-centric classification does not invalidate our findings. Both perspectives converge on the same conclusions.

---
## Summary of Findings

### Hypothesis Results

| Hypothesis | Threshold | Result | Status |
|---|---|---|---|
| **H1:** Traders prefer buying YES | >55% YES buys, p<0.01 | 51.3% (p=0.0001) | **Not confirmed.** Effect exists but below threshold. Market microstructure limits trade-count ratios. |
| **H2:** YES tokens are overpriced | Overpricing > 5pp | Underpricing in our sample | **Not confirmed.** Contradicts Deleep et al. (5,456 markets). Likely sample size limitation. |
| **H3:** Small/large traders differ | >10pp gap | 19pp gap (56% vs 37%) | **Technically confirmed,** but the gradient is primarily a price composition effect, not behavioral preference. |

### Key contextual finding

Single-market events resolve YES 51% of the time — near 50/50. The commonly cited "80% NO" statistic is an artifact of counting markets within multi-market events.

### Tangential findings (not pre-hypothesized, require further work)

- **T1:** Small trades are the only profitable bucket (+$0.024/trade). Edge shrinks with size.
- **T2:** Small wallets lean YES (~55%), large wallets lean NO (~42%). Consistent with price composition mechanism.

### Limitations

- 88 single-market events from top 500 by volume (not representative of all Polymarket)
- Maker-centric trade classification (taker behavior not directly observed)
- First 200 trades per token (temporal bias toward early-life trading)
- Resolved markets only (open-market dynamics may differ)
- Calibration contradicts larger study — sample size is the most likely explanation
- No category metadata on single-market events

### What would strengthen or change these conclusions

1. **500+ single-market events** would make calibration meaningful and settle the Deleep contradiction
2. **Full trade history** (not 200-trade cap) would eliminate temporal VWAP bias
3. **Taker-side classification** would reveal whether the maker-centric view hides taker-side bias
4. **Category tags** would enable cross-category analysis on single-market data
5. **Market maker identification** would clarify whether the wallet-level gradient reflects strategy or liquidity provision